# 🚀 dlt Quick Start : Ingestion E-Commerce & Découverte de DuckDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_1_quick_start.ipynb)

Bienvenue dans ce premier lab d'initiation à **`dlt` (data load tool)**, l'outil open-source moderne pour l'ingestion de données en Python.

### 🎯 Objectifs de ce lab :
1. Comprendre le rôle et l'intérêt de `dlt` dans la pile moderne de Data Engineering (Modern Data Stack).
2. Construire et exécuter votre premier pipeline d'ingestion sur des données e-commerce semi-structurées.
3. Observer la normalisation automatique : comment `dlt` infère les schémas et aplatit les dictionnaires imbriqués.
4. Explorer les données chargées dans **DuckDB** via 3 méthodes (l'accesseur `dataset()`, SQL natif DuckDB, et `sql_client`).


## 📖 Qu'est-ce que `dlt` ?

Dans le monde de la data, extraire des données depuis des APIs REST, des fichiers JSON ou des webhooks pour les charger dans un Data Warehouse (BigQuery, Snowflake, DuckDB) est historiquement un travail fastidieux :
- Les schémas changent continuellement (nouveaux attributs, formats hétérogènes).
- Les données sont souvent imbriquées (nested JSON).
- Écrire des requêtes `CREATE TABLE` manuelles et des scripts de parsing est fragile et répétitif.

**`dlt` automatise tout cela** :
- **Inférence automatique de schéma** : il analyse les données à la volée et crée les tables et colonnes avec les bons types.
- **Normalisation intelligente** : il aplatit les dictionnaires et transforme les listes imbriquées en sous-tables relationnelles.
- **Léger et universel** : s'exécute partout où Python tourne (localement, dans Cloud Run, GitHub Actions ou Google Colab).


 [Documentation ici](https://dlthub.com/docs/reference/explainers/how-dlt-works)

---
## 📦 1. Installation de `dlt` et du moteur DuckDB

Pour ce lab d'introduction, nous utilisons **DuckDB** comme destination de stockage. DuckDB est un moteur SQL analytique (OLAP) ultra-rapide fonctionnant en mémoire ou sur un simple fichier local, sans aucun serveur à installer.

> **Note :** En production, la destination sera simplement remplacée par `bigquery` sans changer le reste du code Python !


In [ ]:
!uv add  "dlt[duckdb]"


Vérifier que le pacquet soit bien installé

---
## 🛒 2. Définir des données e-commerce semi-structurées

Dans le cadre de notre fil rouge e-commerce (TheLook), commençons par un petit jeu de données de commandes.
Remarquez que le champ `customer` est un **dictionnaire imbriqué** contenant le nom et la ville du client. C'est le format classique renvoyé par les APIs modernes.


In [2]:
# Données brutes de commandes avec informations client imbriquées (nested dict)
orders_data = [
    {
        "order_id": 1001,
        "customer": {"name": "Alice Martin", "city": "Lyon"},
        "amount": 89.90,
        "status": "shipped"
    },
    {
        "order_id": 1002,
        "customer": {"name": "Bob Dupont", "city": "Paris"},
        "amount": 149.00,
        "status": "delivered"
    },
    {
        "order_id": 1003,
        "customer": {"name": "Claire Morel", "city": "Marseille"},
        "amount": 34.50,
        "status": "processing"
    },
]

print(f"✅ {len(orders_data)} commandes prêtes pour ingestion.")


✅ 3 commandes prêtes pour ingestion.


---
## ⚙️ 3. Créer et configurer le pipeline `dlt`

Pour déclarer un pipeline avec `dlt`, on utilise la fonction `dlt.pipeline()` avec trois arguments essentiels :
* **`pipeline_name`** : Le nom identifiant votre pipeline (utilisé pour nommer le fichier DuckDB et sauvegarder l'état).
* **`destination`** : Le système cible où charger la data (`"duckdb"`, `"bigquery"`, `"postgres"`, etc.).
* **`dataset_name`** : Le nom du schéma ou groupe de tables (équivalent au Dataset BigQuery).


In [3]:
import dlt

# Initialisation du pipeline vers la destination DuckDB
pipeline = dlt.pipeline(
    pipeline_name="quick_start_ecommerce",
    destination="duckdb",
    dataset_name="ecommerce_raw",
)

print(f"Pipeline initialisé : {pipeline.pipeline_name} (destination: {pipeline.destination})")


Pipeline initialisé : quick_start_ecommerce (destination: <dlt.destinations.duckdb(destination_type='duckdb', staging_dataset_name_layout='%s_staging', enable_dataset_name_normalization=True, info_tables_query_threshold=1000, truncate_tables_on_staging_destination_before_load=True, local_dir='/Users/macbook/Development/cours_lasalle/big_data/02_cours/day1_ingestion/labs', pipeline_name='quick_start_ecommerce', pipeline_working_dir='/Users/macbook/.dlt/pipelines/quick_start_ecommerce', create_indexes=False)>)


---
## 🚀 4. Exécuter l'ingestion (`run()`)

Pour lancer le transfert, on appelle la méthode `pipeline.run()`.
On lui fournit la source de données (`orders_data`) et le nom de la table cible (`table_name="orders"`).


In [4]:
# Exécution du pipeline et capture du rapport d'ingestion
load_info = pipeline.run(orders_data, table_name="orders")

# Affichage du rapport de chargement
print(load_info)


Pipeline quick_start_ecommerce load step finished in 0.06 seconds
1 load package(s) were loaded to destination duckdb and into dataset ecommerce_raw
The duckdb destination used duckdb:////Users/macbook/Development/cours_lasalle/big_data/02_cours/day1_ingestion/labs/quick_start_ecommerce.duckdb location to store data
Load package 1791370689.654922 is LOADED and contains no failed jobs


### 💡 Que vient-il de se passer en coulisses ?

1. `dlt` a inspecté chaque document JSON de la liste.
2. Il a détecté automatiquement les types : `order_id` (entier), `amount` (flottant), `status` (texte).
3. **Aplatissement automatique (*Flattening*) :** Le dictionnaire imbriqué `customer: {"name": ..., "city": ...}` a été automatiquement aplati en deux colonnes distinctes : `customer__name` et `customer__city`.
4. Il a créé la table `orders` dans DuckDB et y a inséré les lignes, sans que nous n'ayons écrit une seule ligne de SQL DDL (`CREATE TABLE`).


---
## 🧪 5. Le paramètre `dev_mode` pour le prototypage rapide

En phase de test, on souhaite souvent rejouer un pipeline sans accumuler de doublons ni garder les anciens schémas.
Le paramètre `dev_mode=True` permet d'isoler chaque exécution en créant un schéma horodaté tout neuf.


In [5]:
# Création d'un pipeline en mode développement
dev_pipeline = dlt.pipeline(
    pipeline_name="ecommerce_dev",
    destination="duckdb",
    dataset_name="ecommerce_raw",
    dev_mode=True,  # Réinitialise le dataset à chaque exécution
)

load_info = dev_pipeline.run(orders_data, table_name="orders")
print(load_info)


Pipeline ecommerce_dev load step finished in 0.06 seconds
1 load package(s) were loaded to destination duckdb and into dataset ecommerce_raw_20261007105809
The duckdb destination used duckdb:////Users/macbook/Development/cours_lasalle/big_data/02_cours/day1_ingestion/labs/ecommerce_dev.duckdb location to store data
Load package 1791370689.784745 is LOADED and contains no failed jobs


---
## 🔍 6. Explorer les données chargées

`dlt` propose plusieurs façons simples d'explorer et de vérifier le résultat de l'ingestion.

### Méthode 1 : L'accesseur `pipeline.dataset()` vers Pandas
Cette méthode est idéale en Data Science et en notebook pour récupérer immédiatement un DataFrame.


In [ ]:
# Récupération du dataset et conversion de la table 'orders' en DataFrame
dataset = dev_pipeline.dataset()
df_orders = dataset.orders.df()


,order_id,amount,status,customer__name,customer__city,_dlt_load_id,_dlt_id
0,1001,89.9,shipped,Alice Martin,Lyon,1791370689.784745,BWo5V2CG1ugQSw
1,1002,149.0,delivered,Bob Dupont,Paris,1791370689.784745,BUW+AxHiSbajTg
2,1003,34.5,processing,Claire Morel,Marseille,1791370689.784745,zrFdQ9HyZHPaIw


In [ ]:
# Affichage des premières lignes
df_orders

# Vérification du type df_orders


---
### Méthode 2 : Connexion native DuckDB

Puisque la destination est DuckDB, un fichier local `ecommerce_dev.duckdb` a été créé.
Nous pouvons nous y connecter directement avec le pilote natif `duckdb` pour inspecter les tables système et le schéma généré.


In [7]:
import duckdb

# Connexion directe à la base DuckDB locale
conn = duckdb.connect(f"{dev_pipeline.pipeline_name}.duckdb")
conn.sql(f"SET search_path = '{dev_pipeline.dataset_name}'")

# Liste de toutes les tables créées
print("Tables présentes dans le dataset :")
display(conn.sql("SHOW TABLES").df())


Tables présentes dans le dataset :


,name
0,_dlt_loads
1,_dlt_pipeline_state
2,_dlt_version
3,orders


Remarquez la présence de 3 tables système préfixées par `_dlt_` :
- `_dlt_loads` : Journal de bord de chaque exécution de pipeline (statut, horodatage).
- `_dlt_pipeline_state` : État interne du pipeline (utile pour les chargements incrémentaux).
- `_dlt_version` : Historique des versions du schéma.

Inspectons la structure technique de notre table `orders` :


In [8]:
# Structure et types de la table orders
conn.sql("DESCRIBE orders").df()


,column_name,column_type,null,key,default,extra
0,order_id,BIGINT,YES,None,None,None
1,amount,DOUBLE,YES,None,None,None
2,status,VARCHAR,YES,None,None,None
3,customer__name,VARCHAR,YES,None,None,None
4,customer__city,VARCHAR,YES,None,None,None
5,_dlt_load_id,VARCHAR,NO,None,None,None
6,_dlt_id,VARCHAR,NO,None,None,None


Interrogeons maintenant la table `orders` en SQL classique. Notez les colonnes aplaties `customer__name` et `customer__city` :


In [9]:
# Requête SQL sur les colonnes métiers
query = """
SELECT 
    order_id,
    customer__name,
    customer__city,
    amount,
    status
FROM orders
"""
conn.sql(query).df()


,order_id,customer__name,customer__city,amount,status
0,1001,Alice Martin,Lyon,89.9,shipped
1,1002,Bob Dupont,Paris,149.0,delivered
2,1003,Claire Morel,Marseille,34.5,processing


---
### Méthode 3 : Le client `sql_client()` universel de `dlt`

Pour écrire du code indépendant de la base cible (valable aussi bien sur DuckDB, PostgreSQL que sur **BigQuery**), `dlt` propose un wrapper universel : `pipeline.sql_client()`.

Il permet d'exécuter des requêtes avec un curseur sécurisé sans se soucier du dialecte ou des identifiants de connexion.


In [10]:
# Exécution d'une requête analytique avec le sql_client de dlt
with dev_pipeline.sql_client() as client:
    sql = """
    SELECT 
        customer__city,
        COUNT(*) AS nb_commandes,
        ROUND(SUM(amount), 2) AS total_ca
    FROM orders
    GROUP BY customer__city
    ORDER BY total_ca DESC
    """
    with client.execute_query(sql) as cursor:
        kpi_df = cursor.df()

kpi_df


,customer__city,nb_commandes,total_ca
0,Paris,1,149.0
1,Lyon,1,89.9
2,Marseille,1,34.5


---
## 🏋️ Exercice d'application : Schema Evolution automatique

Imaginez que l'application e-commerce évolue : désormais, les nouvelles commandes incluent des détails de livraison sous forme d'un objet imbriqué `delivery: {"carrier": "Chronopost", "tracking": "FR8921"}`.

### Votre mission :
1. Définissez une liste `new_orders` avec 2 nouvelles commandes contenant ce nouvel objet `delivery`.
2. Exécutez `dev_pipeline.run(new_orders, table_name="orders")`.
3. Avec le dataframe, observez comment `dlt` a automatiquement fait évoluer le schéma (ajout des colonnes `delivery__carrier` et `delivery__tracking` avec des valeurs `None/NaN` pour les anciennes lignes).


In [ ]:
# 1. Définissez les nouvelles commandes avec l'objet imbriqué 'delivery'
new_orders = []

In [ ]:
# 2. Exécutez le chargement dans la même table 'orders'
dev_pipeline.run(new_orders, table_name="LA_TABLE")

# 3. Affichez la table résultante en dataframe

---
### 👏 Félicitations ! Vous maîtrisez les fondamentaux de `dlt` :
- Création d'un pipeline avec `dlt.pipeline()`.
- Ingestion et typage automatique avec `pipeline.run()`.
- Aplatissement automatique des JSON imbriqués.
- Évolution de schéma sans interruption de service.
